# Ownership & Borrowing

Welcome! Ownership is Rust's most distinctive feature — and often the trickiest for newcomers. Think of it as a strict but fair rulebook: every piece of data has exactly one owner, and the compiler enforces it. No garbage collector, no manual `free()` — and no use-after-free bugs. Once you get it, you'll appreciate the safety it provides.

## The Three Ownership Rules

1. **Each value has exactly one owner.**
2. **When the owner goes out of scope, the value is dropped (freed).**
3. **You can have either one mutable reference or many immutable references, but not both at once.**

These rules are checked at compile time. Violate them, and the compiler won't let your code run.

## Stack vs Heap

Rust values live in one of two places:
- **Stack**: Fixed-size data, fast allocation. Local variables, primitives, small structs.
- **Heap**: Dynamic-size data, slower. `String`, `Vec`, and other growable types.

Ownership determines who is responsible for freeing heap memory when it's no longer needed.

In [ ]:
let x = 5;           // Stack: i32
let s = String::from("hello");  // Heap: String owns its buffer
println!("x = {}, s = {}", x, s);

## Move Semantics

When you assign a heap-allocated value to another variable, **ownership moves**. The original variable is no longer valid — it's like handing over a key to a house. You can't use the old variable after the move.

In [ ]:
let s1 = String::from("hello");
let s2 = s1;  // s1's ownership moves to s2
// println!("{}", s1);  // ERROR: s1 is no longer valid!
println!("s2 = {}", s2);

## Copy Types vs Non-Copy

Types that implement `Copy` are **copied** instead of moved. Primitives like `i32`, `bool`, `char` are `Copy`. `String`, `Vec`, and most heap-allocated types are **not** `Copy` — they use move semantics.

In [ ]:
let x = 42;
let y = x;  // x is COPIED, both are valid
println!("x = {}, y = {}", x, y);

let s = String::from("hi");
let t = s.clone();  // Explicit clone: both s and t are valid
println!("s = {}, t = {}", s, t);

## References: Borrowing with &amp;T and &amp;mut T

Instead of moving, you can **borrow** with a reference. `&T` is an immutable reference (read-only). `&mut T` is a mutable reference (can modify). The borrower doesn't own the data — the owner still does.

In [ ]:
fn calculate_length(s: &String) -> usize {
    s.len()  // We borrow, we don't take ownership
}

let s = String::from("hello");
let len = calculate_length(&s);
println!("'{}' has length {}", s, len);  // s is still valid!

In [ ]:
fn append_world(s: &mut String) {
    s.push_str(" world");
}

let mut s = String::from("hello");
append_world(&mut s);
println!("{}", s);

## The Borrowing Rules

1. **Many immutable references** (`&T`) are allowed at the same time.
2. **Only one mutable reference** (`&mut T`) at a time — and no immutable refs while it exists.
3. References must always be valid (no dangling references).

This prevents data races and use-after-free at compile time!

In [ ]:
let s = String::from("hello");
let r1 = &s;
let r2 = &s;  // Multiple immutable refs: OK!
println!("r1: {}, r2: {}", r1, r2);

In [ ]:
let mut s = String::from("hello");
let r1 = &mut s;
r1.push_str("!");
// let r2 = &mut s;  // ERROR: can't have two mutable refs
// let r3 = &s;      // ERROR: can't have immutable while mutable exists
println!("{}", s);

## Dangling References: Why Rust Prevents Them

A dangling reference points to memory that has been freed. Rust's borrow checker ensures references never outlive their data. If you try to return a reference to a local variable, the compiler will stop you.

## Slices: &amp;str and &amp;[T]

A **slice** is a reference to a contiguous sequence of elements. `&str` is a string slice (view into UTF-8 text). `&[T]` is a slice of elements of type `T`. Slices don't own data — they borrow it.

In [ ]:
let s = String::from("hello world");
let hello: &str = &s[0..5];   // Slice of first 5 bytes
let world: &str = &s[6..11];  // Slice of "world"
println!("{} | {}", hello, world);

In [ ]:
let arr = [1, 2, 3, 4, 5];
let slice: &[i32] = &arr[1..4];  // [2, 3, 4]
println!("slice: {:?}", slice);

## Ownership in Function Calls and Returns

Passing a value to a function **moves** it (unless it's `Copy` or you pass a reference). Returning a value **moves** it to the caller. Design your functions to take references when you don't need ownership.

In [ ]:
fn take_ownership(s: String) {
    println!("I own: {}", s);
}  // s is dropped here

fn give_ownership() -> String {
    String::from("hello")
}

let s1 = give_ownership();
take_ownership(s1);
// s1 is no longer valid
let s2 = give_ownership();
println!("s2 = {}", s2);

## String vs &amp;str in Practice

- **`String`**: Owned, growable, heap-allocated. Use when you need to build or modify strings.
- **`&str`**: Borrowed slice, immutable. Use for function parameters when you don't need ownership — it accepts both `&str` and `&String` via deref coercion.

Prefer `&str` for function parameters when you only need to read.

In [ ]:
fn print_greeting(name: &str) {
    println!("Hello, {}!", name);
}

let s = String::from("Alice");
print_greeting(&s);       // &String coerces to &str
print_greeting("Bob");     // String literal is &str

## Common Ownership Patterns

1. **Borrow when possible** — use `&T` or `&mut T` to avoid moves
2. **Clone when needed** — use `.clone()` when you truly need a copy
3. **Return owned data** — functions that create data return it (e.g., `String::from`)
4. **Use references in structs** — `struct Foo { s: &str }` requires a lifetime when the struct holds a reference

## Common Mistakes Beginners Make

1. **Using a value after move** — the compiler will tell you; fix by borrowing or cloning
2. **Mixing mutable and immutable borrows** — remember: many shared OR one exclusive
3. **Overusing `.clone()`** — often you can borrow instead
4. **Taking `String` when `&str` would work** — prefer `&str` for read-only parameters

## Key Rules to Remember

- Each value has one owner; ownership moves on assignment (for non-Copy types)
- `&T` = immutable borrow; `&mut T` = mutable borrow
- Borrowing rules: many shared OR one mutable, never both
- Slices (`&str`, `&[T]`) are references; they don't own data
- Prefer `&str` for function parameters when you only need to read
- The borrow checker is your friend — it catches bugs before they run!